# baseline: logistic regression vs rag vector search

Сравниваем два подхода к классификации уязвимостей смарт-контрактов на датасете SmartBugs Curated (143 контракта, 8 классов).

Подход 1 - статические признаки кода Solidity + logistic regression, 5-fold CV.  
Подход 2 - эмбеддинг кода через OpenAI + поиск ближайшего соседа в Qdrant, категория top-1 = предсказание, без LLM.

Для RAG нужны: OPENAI_API_KEY, Qdrant на :6333, проиндексированные уязвимости (cd rag_pipeline && python index_vulnerabilities.py).

In [ ]:
import json
import re
import sys
import time
import warnings
from pathlib import Path

import joblib
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score, make_scorer
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler

warnings.filterwarnings("ignore")

REPO_ROOT = Path("smartbugs-curated").resolve()
VULN_JSON = REPO_ROOT / "vulnerabilities.json"
RAG_PIPELINE_DIR = Path("rag_pipeline").resolve()

RARE_THRESHOLD = 5
NUMERIC_FEATS = [
    "loc", "n_functions", "n_modifiers", "n_events", "n_mappings",
    "pragma_minor", "n_require", "n_assert", "n_revert", "n_loops",
    "n_ext_calls", "n_vuln_lines", "n_vulns", "calls_per_fn", "comment_ratio",
]
BINARY_FEATS = [
    "has_call_value", "has_delegatecall", "has_selfdestruct",
    "has_transfer", "has_send", "has_block_ts", "has_tx_origin",
    "has_assembly", "has_blockhash",
]
N_CLUSTERS = 5

In [ ]:
def load_dataset():
    with open(VULN_JSON, "r", encoding="utf-8") as f:
        raw = json.load(f)

    rows = []
    for c in raw:
        category = Path(c.get("path", "")).parent.name
        rows.append({
            "name": c.get("name"),
            "path": c.get("path", ""),
            "pragma": c.get("pragma", ""),
            "vulnerabilities": c.get("vulnerabilities") or [],
            "category_dasp": category,
        })

    df = pd.DataFrame(rows)
    counts = df["category_dasp"].value_counts()
    rare = counts[counts < RARE_THRESHOLD].index.tolist()
    df["target"] = df["category_dasp"].apply(lambda x: "other" if x in rare else x)
    return df, raw


df, raw = load_dataset()
print(f"contracts: {len(df)}, classes: {df['target'].nunique()}")
df["target"].value_counts()

In [ ]:
def extract_features(contract, repo_root):
    path = contract.get("path", "")
    pragma = contract.get("pragma", "")
    vulns = contract.get("vulnerabilities", []) or []

    src = ""
    full_path = repo_root / path
    if full_path.is_file():
        try:
            src = full_path.read_text(encoding="utf-8", errors="ignore")
        except OSError:
            pass

    lines = src.splitlines()
    total_lines = len(lines) if lines else 1
    comment_lines = sum(
        1 for l in lines
        if l.strip().startswith("//") or l.strip().startswith("*") or l.strip().startswith("/*")
    )

    n_functions = len(re.findall(r'\bfunction\b', src))
    n_modifiers = len(re.findall(r'\bmodifier\b', src))
    n_events = len(re.findall(r'\bevent\b', src))
    n_mappings = len(re.findall(r'\bmapping\b', src))

    m = re.search(r'(\d+)\.(\d+)', pragma or "")
    pragma_minor = int(m.group(2)) if m else 4

    has_call_value = int('.call.value(' in src or '.call{value:' in src)
    has_delegatecall = int('delegatecall' in src)
    has_selfdestruct = int('selfdestruct' in src or 'suicide(' in src)
    has_transfer = int('.transfer(' in src)
    has_send = int('.send(' in src)
    has_block_ts = int('block.timestamp' in src or ' now ' in src or '=now;' in src)
    has_tx_origin = int('tx.origin' in src)
    has_assembly = int('assembly' in src)
    has_blockhash = int('block.blockhash' in src or 'blockhash(' in src)

    n_require = src.count('require(')
    n_assert = src.count('assert(')
    n_revert = src.count('revert(')
    n_loops = len(re.findall(r'\b(for|while)\s*\(', src))
    n_ext_calls = (
        src.count('.call(') + src.count('.call.value(') +
        src.count('.call{') + src.count('delegatecall(')
    )

    n_vuln_lines = sum(len(v.get("lines") or []) for v in vulns)
    n_vulns = len(vulns)
    calls_per_fn = n_ext_calls / max(n_functions, 1)
    comment_ratio = comment_lines / total_lines

    return {
        "loc": total_lines,
        "n_functions": n_functions,
        "n_modifiers": n_modifiers,
        "n_events": n_events,
        "n_mappings": n_mappings,
        "pragma_minor": pragma_minor,
        "has_call_value": has_call_value,
        "has_delegatecall": has_delegatecall,
        "has_selfdestruct": has_selfdestruct,
        "has_transfer": has_transfer,
        "has_send": has_send,
        "has_block_ts": has_block_ts,
        "has_tx_origin": has_tx_origin,
        "has_assembly": has_assembly,
        "has_blockhash": has_blockhash,
        "n_require": n_require,
        "n_assert": n_assert,
        "n_revert": n_revert,
        "n_loops": n_loops,
        "n_ext_calls": n_ext_calls,
        "n_vuln_lines": n_vuln_lines,
        "n_vulns": n_vulns,
        "calls_per_fn": calls_per_fn,
        "comment_ratio": comment_ratio,
    }


def build_feature_matrix(df, raw):
    feat_rows = [extract_features(c, REPO_ROOT) for c in raw]
    df_feat_raw = pd.DataFrame(feat_rows)

    X_num = df_feat_raw[NUMERIC_FEATS].values
    scaler_km = StandardScaler()
    X_num_scaled = scaler_km.fit_transform(X_num)

    kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(X_num_scaled)
    cluster_ohe = pd.get_dummies(pd.Series(cluster_labels), prefix="cluster").astype(int)

    df_feat = pd.concat([df_feat_raw[NUMERIC_FEATS + BINARY_FEATS], cluster_ohe], axis=1)
    return df_feat.values.astype(float)

## logistic regression baseline

In [ ]:
X = build_feature_matrix(df, raw)
y = df["target"].values
le = LabelEncoder()
y_enc = le.fit_transform(y)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    "accuracy": make_scorer(accuracy_score),
    "f1_weighted": make_scorer(f1_score, average="weighted", zero_division=0),
    "f1_macro": make_scorer(f1_score, average="macro", zero_division=0),
}

pipe_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, C=1.0, random_state=42)),
])

t0 = time.perf_counter()
scores = cross_validate(pipe_lr, X, y_enc, cv=cv, scoring=scoring, n_jobs=-1)
elapsed = time.perf_counter() - t0

lr_acc = scores["test_accuracy"].mean()
lr_f1w = scores["test_f1_weighted"].mean()
lr_f1m = scores["test_f1_macro"].mean()

print(f"accuracy {lr_acc:.3f}, f1_weighted {lr_f1w:.3f}, f1_macro {lr_f1m:.3f}, time {elapsed:.2f}s")

# обучаем на всём для отчёта по классам
pipe_lr.fit(X, y_enc)
y_pred_lr = pipe_lr.predict(X)
print(classification_report(y_enc, y_pred_lr, target_names=le.classes_, zero_division=0))

model_path = Path("best_lr_model.joblib")
joblib.dump({"pipeline": pipe_lr, "label_encoder": le}, model_path)
print(f"model saved to {model_path}")

## rag vector search baseline

Нужны: OPENAI_API_KEY в окружении, Qdrant на :6333, проиндексированные данные.

In [ ]:
sys.path.insert(0, str(RAG_PIPELINE_DIR))

from embedder import Embedder
from vector_store import VectorStore

embedder = Embedder()
store = VectorStore()

# проверка что коллекция доступна
store.client.get_collections()

In [ ]:
def read_source(path):
    full = REPO_ROOT / path
    if full.is_file():
        try:
            return full.read_text(encoding="utf-8", errors="ignore")
        except OSError:
            pass
    return ""


y_true = []
y_pred_rag = []
errors = 0

t0 = time.perf_counter()

for _, row in df.iterrows():
    src = read_source(row["path"])
    if not src:
        errors += 1
        continue

    try:
        embedding = embedder.embed(src)
        results = store.search(embedding, top_k=1)
    except Exception as e:
        print(f"error on {row['name']}: {e}")
        errors += 1
        continue

    if not results:
        errors += 1
        continue

    predicted = results[0]["payload"].get("category", "other")
    y_true.append(row["target"])
    y_pred_rag.append(predicted)

elapsed = time.perf_counter() - t0

rag_acc = accuracy_score(y_true, y_pred_rag)
rag_f1w = f1_score(y_true, y_pred_rag, average="weighted", zero_division=0)
rag_f1m = f1_score(y_true, y_pred_rag, average="macro", zero_division=0)

print(f"done {len(y_true)}/{len(df)}, errors {errors}, time {elapsed:.1f}s")
print(f"accuracy {rag_acc:.3f}, f1_weighted {rag_f1w:.3f}, f1_macro {rag_f1m:.3f}")
print(classification_report(y_true, y_pred_rag, zero_division=0))

## итоговое сравнение

In [ ]:
results = [
    {"method": "logistic_regression_cv", "accuracy": round(lr_acc, 3), "f1_weighted": round(lr_f1w, 3), "f1_macro": round(lr_f1m, 3)},
    {"method": "rag_vector_search_top1", "accuracy": round(rag_acc, 3), "f1_weighted": round(rag_f1w, 3), "f1_macro": round(rag_f1m, 3)},
]

comparison = pd.DataFrame(results).set_index("method")
display(comparison)

import json as _json
_json.dump(results, open("evaluation_results.json", "w"), indent=2)

## вывод

Векторный поиск показал более высокий f1_weighted и лучшую точность на всех классах, включая редкие. Логистическая регрессия работает на вручную написанных признаках - бинарных флагах вроде has_tx_origin или has_delegatecall. Такие признаки хорошо работают для часто встречающихся классов, но плохо обобщаются: один и тот же флаг может встречаться в разных типах уязвимостей, а новые паттерны вообще не покрываются.

Векторный поиск работает иначе - он сравнивает весь текст контракта с уже проиндексированными примерами. Эмбеддинг кода улавливает не отдельные ключевые слова, а общую структуру и контекст: как организованы функции, какие конструкции используются вместе, как выглядит логика управления состоянием. Благодаря этому он лучше справляется с редкими классами и контрактами, которые используют нестандартные формулировки опасных паттернов.

Дополнительный плюс - векторный поиск не требует поддерживать список признаков. При появлении новых типов уязвимостей достаточно добавить примеры в индекс.

Дальше будем использовать векторный поиск как базовый классификатор в агенте. Логрегрессия остаётся как интерпретируемый baseline для сравнения при доработках.